In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TextClassificationPipeline
import joblib
from datasets import Dataset
from transformers import DataCollatorWithPadding
import torch
from sklearn.utils import shuffle


C:\Users\sungj\Documents\GitHub\Atlantic-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained("finiteautomata/bertweet-base-sentiment-analysis", use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(
    "finiteautomata/bertweet-base-sentiment-analysis",
    id2label = {0: 'Negative', 1: 'Neutral', 2: 'Positive'},
    label2id = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12565.28it/s]


In [51]:
df = pd.read_csv("processed_sentiment_data.csv")

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

In [31]:
def tokenize_fn(batch):
  return tokenizer(
      batch["text"],
      truncation = True,
      max_length= 128,
      return_tensors = "pt",
      padding = True
  )
tokenized_dataset = dataset.map(tokenize_fn, batched = True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("sentiment", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer) #data collator forms batches using the dataset

Map: 100%|██████████| 62483/62483 [00:45<00:00, 1376.52 examples/s]


In [32]:
train_ds = tokenized_dataset["train"].shuffle(seed=42).select(range(len(tokenized_dataset["train"])))
eval_ds  = tokenized_dataset["test"].shuffle(seed=42).select(range(len(tokenized_dataset["test"])))

In [5]:
# !pip install -q evaluate scikit-learn
# !pip install -q -U torchvision

In [ ]:
import evaluate
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

#https://deepwiki.com/huggingface/transformers/3.2-trainingarguments-configuration
#This website has a ton of information about the different parameters.
training_args = TrainingArguments(
    output_dir="./roberta-sentiment-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    #per_device_train_batch_size controls training batch size. per_device_eval_batch_size evaluation batch size.
    learning_rate=.00004,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs=5,
    logging_steps=50,
    weight_decay=0.01, #this adds a L2 penalty
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback()]
    #https://github.com/huggingface/transformers/blob/v5.14.0/src/transformers/trainer_callback.py#L711
    #https://huggingface.co/docs/transformers/v5.14.0/en/main_classes/callback#transformers.EarlyStoppingCallback
)
trainer.train()

In [ ]:
metrics = trainer.evaluate()
print(metrics)
from sklearn.metrics import classification_report
preds_output = trainer.predict(eval_ds)
y_pred = np.argmax(preds_output.predictions, axis =-1)
y_true = preds_output.label_ids
print(classification_report(y_true, y_pred))

In [ ]:
trainer.save_model("roberta_sentiment_model")
tokenizer.save_pretrained("roberta_sentiment_model", use_fast=False) #I was wondering why my tokenizer kept on not working, and after searching it up, it said that it can return a different result when you use the fast which is uses a Rust backend. This was causing the problem.

In [52]:
#I didnt now how to do the actual predictions part and I wanted to find an easy way to use the tokenizer and model. Using this text classification pipeline makes it easy to combine it, instead of making a predict method.
#https://discuss.huggingface.co/t/i-have-trained-my-classifier-now-how-do-i-do-predictions/3625
roberta_model = TextClassificationPipeline(
    model = model,
    tokenizer = tokenizer,
)

In [111]:
from logreg_class import LogRegTfidfModel
from finetuned_roberta_class import FTRobertaModel

logreg_model = LogRegTfidfModel()
fine_tuned_model = FTRobertaModel()

def predict(text):
    print(text)

    logreg_model.predict(text)
    #logreg_model.print_prediction()

    fine_tuned_model.predict(text)
    #fine_tuned_model.print_prediction()

    r_pred = roberta_model.predict(text)
    state = 2
    if r_pred[0]['label'] == "Negative":
        state = 0
    elif r_pred[0]['label'] == "Neutral":
        state = 1

    print(logreg_model.pred_number, fine_tuned_model.pred_number, state)
    # print(f"\n** Roberta Model **\nPredicted Sentiment: {r_pred[0]['label']}\nConfidence: %{(r_pred[0]['score'] * 100):.2f}")
    return logreg_model.pred_number, fine_tuned_model.pred_number, state


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1686.34it/s]


In [115]:
def accuracy_models(acc_df, iters=10, start_iter=0):
    f_count = 0
    r_count = 0
    l_count = 0
    too_long_idx = []


    for i in np.arange(start_iter, start_iter+iters, 1):
        try:
            l_pred, f_pred, r_pred = predict(acc_df.loc[i, "processed_text"])

            act = acc_df.loc[i, "sentiment"]

            if f_pred == act:
                f_count += 1
            if r_pred == act:
                r_count += 1
            if l_pred == act:
                l_count += 1

        except RuntimeError:
            too_long_idx.append(i)

    acc_df = acc_df.drop(index=too_long_idx).reset_index(drop=True)
    print(f"LogReg Accuracy: %{(l_count / (iters - len(too_long_idx)) * 100):.2f}")
    print(f"Roberta Accuracy: %{(r_count / (iters - len(too_long_idx)) * 100):.2f}")
    print(f"FineTuned Roberta Accuracy: %{(f_count / (iters - len(too_long_idx)) * 100):.2f}")
    print(f"# of bad idxs: {len(too_long_idx)}")
    return acc_df

In [116]:
test_df = pd.DataFrame(dataset["test"])

In [118]:
accuracy_models(test_df, 5000)
#df.to_csv("processed_sentiment_data.csv", index=False)

lovely view cheap last minute room rate great location
2 tensor(2) 2
recently good friend call need something never say thank kind word
2 tensor(2) 2
half screen stop working easy change change screen need follow youtube tutorial change broken screen cell phone around month ago still work perfectlybr br update purchase screen july get july screen easy replace work good past month unfortunately week month later half screen stop work blue since replacement always wear case phone well screen protector anyways not drop phone neither anything may cause screen quotfrozenquot use phone usually place pocket later pick phone screen frozen see picture research online issue not find thing specific problem figure problem may relate manufacture screen make poorly also touch screen work entire screen problem image display big problemi disappoint not lot buying option phone screen replacement one expansive not work long time
0 tensor(0) 0
report dishwasher malfunction everything go smoothly
0 tensor(

,processed_text,sentiment
0,lovely view cheap last minute room rate great ...,2
1,recently good friend call need something never...,0
2,half screen stop working easy change change sc...,0
3,report dishwasher malfunction everything go sm...,0
4,duty manager night take rude behave unexceptab...,0
...,...,...
59626,product quality meet basic expectation average...,1
59627,great render animation purchase processor rend...,2
59628,not take laundry two day call reception,0
59629,plug play fast delivery super light yet feels ...,2


In [121]:
def pred(text):
    logreg_model.predict(text)
    fine_tuned_model.predict(text)
    print(logreg_model.pred_label, fine_tuned_model.pred_label)